[Reference](https://ai.plainenglish.io/build-your-own-code-interpreter-a-secure-docker-sandbox-for-llm-agents-6cc0d85e9c95)

# Step 1: The Secure Environment (Dockerfile)
```
FROM python:3.12-slim

# 1. Install uv for lightning-fast package management
COPY --from=ghcr.io/astral-sh/uv:latest /uv /bin/uv

# 2. Setup user permissions (Security Best Practice)
RUN useradd -m codeuser && \
    mkdir /app && \
    chown codeuser:codeuser /app

USER codeuser
WORKDIR /app

# 3. Configure Virtual Environment
ENV VIRTUAL_ENV=/app/.venv \
    PATH="/app/.venv/bin:$PATH"

# 4. Create venv and pre-install common libs
RUN uv venv && \
    uv pip install --no-cache requests numpy

# 5. The Wrapper Script
COPY --chown=codeuser:codeuser src/exec_script.py /app/

# The container acts as an executable wrapper
ENTRYPOINT ["python", "exec_script.py"]
```


# Step 2: The “Magic” Wrapper Script

In [2]:
# src/exec_script.py
import sys
import subprocess
import re
import importlib
import json

def install_packages(code_str: str):
    """
    Scans code for '# AUTO_INSTALL: pandas numpy' and installs them using uv.
    """
    pattern = r'#\s*AUTO_INSTALL:\s*(.+)'
    matches = re.findall(pattern, code_str)

    if matches:
        packages = []
        for match in matches:
            packages.extend(match.split())

        # Use uv for fast installation
        print(json.dumps({"status": "info", "output": f"Installing packages: {' '.join(packages)}"}))
        subprocess.check_call(["uv", "pip", "install"] + packages)
        print(json.dumps({"status": "info", "output": "Packages installed successfully"}))
        importlib.invalidate_caches()

def execute_code(code_str: str):
    try:
        install_packages(code_str)

        # Execute the code in a shared scope
        execution_scope = {}
        exec(code_str, execution_scope, execution_scope)

        # Extract the 'result' variable
        result = execution_scope.get('result', None)
        print(json.dumps({"status": "success", "output": result}))

    except Exception as e:
        print(json.dumps({"status": "error", "output": str(e)}))

if __name__ == "__main__":
    # Read the code file passed from Docker command
    with open(sys.argv[1], 'r') as f:
        code = f.read()
    execute_code(code)

# Step 3: The Code Extractor

In [3]:
# src/extract_code.py
def extract_code(response: str) -> str:
    code = response
    if "```python" in code:
        code = code.split("```python")[1].split("```")[0].strip()
    elif "```" in code:
        code = code.split("```")[1].split("```")[0].strip()
    return code

# Step 4: Bridging Python and Docker

In [4]:
import docker
import tempfile
import os

def run_code_in_container(code: str) -> str:
    # 1. Write the LLM code to a temp file on the host
    with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False, dir='/tmp') as tmp:
        tmp.write(code)
        code_path = tmp.name

    try:
        client = docker.from_env()

        # 2. Run the container
        # The ENTRYPOINT is 'python exec_script.py', so we just pass the file path
        container = client.containers.run(
            'langchain-sandbox',
            command=[f'/code/{os.path.basename(code_path)}'],
            volumes={'/tmp': {'bind': '/code', 'mode': 'ro'}}, # Mount host /tmp to container /code
            network_mode='bridge',  # Allow internet for pip install
            mem_limit='512m',       # Sandbox resource limits
            remove=True             # Auto-delete after run
        )

        result= container.decode('utf-8').strip()
    except Exception as e:
        result = f"""{{"status": "error", "output": {str(e)}}}"""
    finally:
        try:
            os.unlink(code_path) # Cleanup host file
        except:
            pass

    return result

# Step 5: The Agent (Main Logic)

In [5]:
# main.py
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.messages import HumanMessage, SystemMessage
from src.exec_code import run_code_in_container
from src.extract_code import extract_code

from dotenv import load_dotenv
load_dotenv()

# Setup LLM
llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen3-Coder-480B-A35B-Instruct",
    task="text-generation",
    max_new_tokens=512
)
chat_model = ChatHuggingFace(llm=llm)

# The Prompt Engineering
system_prompt = """
generate python code to find answer.
Instructions:
1. Add '# AUTO_INSTALL: package_name' for dependencies.
2. Put logic in a function.
3. The final answer should be a descriptive string.
4. Store final answer in a variable named 'result'.
"""

def chat(query: str):
    # 1. Ask LLM to generate code
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=query)
    ]
    response = chat_model.invoke(messages)

    # 2. Extract and Run Code
    code = extract_code(response.content)
    print(f"Generated code:\n{code}")

    output = run_code_in_container(code)
    print(f"Result: {output}")

# Run it!
chat("what's the time and date now in Indian timezone")